# Чат-агент на LangChain с памятью в файле

In [1]:
import os
import sys
import json
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

from src.file_fact_memory import FileFactMemory
from src.display_cell import render_cell
from src.fact_extractor import FactExtractor
from src.langchain_llm import LocalLlamaChatModel
from src.agent import ToolAgent

## Блок 1. Служебная CELL

In [2]:
render_cell(
    title="СЛУЖЕБНЫЙ БЛОК: ИНИЦИАЛИЗАЦИЯ ЛИЧНОГО АССИСТЕНТА",
    content={
        "Компонент": "DisplayCell",
        "Среда": "Jupyter Notebook / Console",
        "Назначение": "Изолированный вывод служебных логов памяти и действий агента",
        "Статус": "Активен"
    },
    border_style="bold magenta",
    subtitle="Демонстрация собственной CELL"
)

## Блок 2. Цепочка LangChain (LCEL)

In [3]:
lcel_demo_file = "memory_lcel_test.json"
lcel_memory = FileFactMemory(file_path=lcel_demo_file, session_id="test_user", show_service_cell=True)

lcel_memory.facts = {
    "name": "Алексей",
    "city": "Санкт-Петербург",
    "preferences": "Кофе без сахара, краткие ответы",
    "goals": "Инвестиции в акции и недвижимость"
}
lcel_memory.save_to_file()

prompt_template = ChatPromptTemplate.from_messages([
    ("system",
     "Ты — личный ассистент с долговременной памятью.\n"
     "Вот что ты знаешь о пользователе из файла памяти:\n"
     "----------------------------------------\n"
     "{facts}\n"
     "----------------------------------------\n"
     "Отвечай кратко и доброжелательно на основе этих фактов."),
    ("human", "{question}")
])

llm = LocalLlamaChatModel(model_alias="qwen")

chain = (
    {
        "facts": RunnableLambda(lambda _: lcel_memory.load_memory_variables({})["facts"]),
        "question": RunnablePassthrough(),
    }
    | prompt_template
    | llm
    | StrOutputParser()
)

print("\n--- Вызов цепочки LangChain ---")
answer = chain.invoke("В каком городе я живу и какие у меня цели?")
print(f"Ответ цепочки: {answer}")

if os.path.exists(lcel_demo_file):
    os.remove(lcel_demo_file)


--- Вызов цепочки LangChain ---
Ответ цепочки: Привет! Я принял твоё сообщение: 'В каком городе я живу и какие у меня цели?'.


C:\Users\veles\AppData\Local\Programs\Python\Python312\Lib\site-packages\pydantic\main.py:263: LangChainDeprecationWarning: The class `InMemoryChatMessageHistory` was deprecated in LangChain 1.6.4 and will be removed in 2.0.0 See the short-term memory documentation for recommended alternatives: https://docs.langchain.com/oss/python/langchain/short-term-memory
  validated_self = self.__pydantic_validator__.validate_python(data, self_instance=self)


## Блок 3. Сессия 1: Запись фактов

In [4]:
DEMO_FILE = "memory_demo.json"

if os.path.exists(DEMO_FILE):
    os.remove(DEMO_FILE)

SESSION_ID = "matvey_profile"

memory_session_1 = FileFactMemory(file_path=DEMO_FILE, session_id=SESSION_ID, show_service_cell=True)
agent_session_1 = ToolAgent(memory=memory_session_1, session_id=SESSION_ID, file_path=DEMO_FILE, show_cell=True)

greeting_1 = agent_session_1.get_greeting()
print(f"\n[Ассистент]: {greeting_1}\n")

user_msg_1 = "Привет! Меня зовут Матвей, я живу в Москве. Моя цель — инвестиции в акции и криптовалюту."
print(f"[Пользователь]: {user_msg_1}")

res_1 = agent_session_1.run(user_msg_1)
print(f"[Ассистент]: {res_1['final_answer']}\n")

user_msg_2 = "Запомни: я люблю кофе без сахара и предпочитаю краткие ответы по делу."
print(f"[Пользователь]: {user_msg_2}")

res_2 = agent_session_1.run(user_msg_2)
print(f"[Ассистент]: {res_2['final_answer']}\n")


[Ассистент]: Привет! Я твой личный ассистент. Я запоминаю важные факты о тебе в файл memory.json, умею узнавать цены криптовалют (BTC, ETH), акций российских компаний (Сбер, Газпром, Яндекс) и помогать с задачами. Как к тебе обращаться?

[Пользователь]: Привет! Меня зовут Матвей, я живу в Москве. Моя цель — инвестиции в акции и криптовалюту.


[Ассистент]: Твои интересы и цели, Матвей: Инвестиции, криптовалюта и акции.

[Пользователь]: Запомни: я люблю кофе без сахара и предпочитаю краткие ответы по делу.


[Ассистент]: Твои предпочтения, Матвей: Кофе без сахара, краткие ответы.



## Блок 4. Перезапуск программы

In [5]:
del agent_session_1
del memory_session_1

print(f"Статус файла '{DEMO_FILE}' на диске: существует = {os.path.exists(DEMO_FILE)}")

with open(DEMO_FILE, "r", encoding="utf-8") as f:
    saved_facts = json.load(f)

print("\nСодержимое сохранённого файла памяти на диске:")
print(json.dumps(saved_facts, ensure_ascii=False, indent=2))

Статус файла 'memory_demo.json' на диске: существует = True

Содержимое сохранённого файла памяти на диске:
{
  "matvey_profile": {
    "name": "Матвей",
    "city": "Москва",
    "goals": "Инвестиции, криптовалюта и акции",
    "preferences": "Кофе без сахара, краткие ответы"
  }
}


## Блок 5. Сессия 2: Восстановление памяти

In [6]:
memory_session_2 = FileFactMemory(file_path=DEMO_FILE, session_id=SESSION_ID, show_service_cell=True)
agent_session_2 = ToolAgent(memory=memory_session_2, session_id=SESSION_ID, file_path=DEMO_FILE, show_cell=True)

greeting_2 = agent_session_2.get_greeting()
print(f"\n[Ассистент]: {greeting_2}\n")

query_1 = "Как меня зовут и в каком городе я живу?"
print(f"[Пользователь]: {query_1}")
ans_1 = agent_session_2.run(query_1)
print(f"[Ассистент]: {ans_1['final_answer']}\n")

query_2 = "Напомни мои предпочтения и цели?"
print(f"[Пользователь]: {query_2}")
ans_2 = agent_session_2.run(query_2)
print(f"[Ассистент]: {ans_2['final_answer']}\n")


[Ассистент]: Привет, Матвей! Рад снова быть на связи (city: Москва, goals: Инвестиции, криптовалюта и акции, preferences: Кофе без сахара, краткие ответы). Чем могу помочь тебе сегодня?

[Пользователь]: Как меня зовут и в каком городе я живу?
[Ассистент]: Тебя зовут Матвей, и ты живёшь в городе Москва.

[Пользователь]: Напомни мои предпочтения и цели?
[Ассистент]: Твои предпочтения, Матвей: Кофе без сахара, краткие ответы.



## Блок 6. Работа с инструментами

In [7]:
query_tool = "Сколько сейчас стоит биткоин?"
print(f"[Пользователь]: {query_tool}")

res_tool = agent_session_2.run(query_tool)
print(f"[Ассистент]: {res_tool['final_answer']}\n")

query_stock = "Какая текущая цена акций Сбербанка?"
print(f"[Пользователь]: {query_stock}")

res_stock = agent_session_2.run(query_stock)
print(f"[Ассистент]: {res_stock['final_answer']}\n")

print(f"Имя в памяти: {agent_session_2.memory.facts.get('name')}")
print(f"Город в памяти: {agent_session_2.memory.facts.get('city')}")
print(f"Цели в памяти: {agent_session_2.memory.facts.get('goals')}")
print(f"Предпочтения в памяти: {agent_session_2.memory.facts.get('preferences')}")

[Пользователь]: Сколько сейчас стоит биткоин?


[Ассистент]: Биткоин (BTC): $94,850.00 (~8,760,000 RUB)

[Пользователь]: Какая текущая цена акций Сбербанка?


[Ассистент]: Акции Сбербанка (SBER): 312.40 RUB

Имя в памяти: Матвей
Город в памяти: Москва
Цели в памяти: Инвестиции, криптовалюта и акции
Предпочтения в памяти: Кофе без сахара, краткие ответы
